# 策略 × 行业矩阵回测（10年验证）

用 2016-2026 共 10 年数据验证上一轮的结论是否仍然成立。

**策略池**：reversed_gtja_vwap, gtja_vwap, gtja_momentum, gtja_volume_price, gtja_volatility

**股票池**：default.yaml 100 只 CSI 300 成分股，11 个行业分组

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

from src.config.loader import load_config
from src.data.fetcher import fetch_daily
from src.data.storage import save_parquet, load_parquet
from src.data.filters import detect_limit_price, detect_suspension
from src.data.universe import resolve_universe
from src.data import validate_ohlcv
from src.analysis.pool_matrix import run_matrix, pivot_matrix, best_per_pool

print('All imports OK')

All imports OK


## Step 1: 加载数据（10年：2016-05 ~ 2026-05）

In [2]:
cfg = load_config(Path('../configs/default.yaml'))
universe_cfg = cfg.get('universe', {})

STOCKS = resolve_universe(universe_cfg)
START = '2016-05-24'
END = '2026-05-24'
RAW_DIR = Path('../data/raw')

print(f'Loading {len(STOCKS)} stocks from {START} to {END}...')

frames = []
missing_codes = []
for code in STOCKS:
    path = RAW_DIR / f'{code}.parquet'
    if path.exists():
        df = load_parquet(path)
        # 验证日期范围是否覆盖全区间
        if df['date'].min() > pd.Timestamp('2016-06-01'):
            df = fetch_daily(code, START, END)
            save_parquet(df, path)
    else:
        try:
            df = fetch_daily(code, START, END)
            save_parquet(df, path)
        except Exception as e:
            print(f'  Failed to fetch {code}: {e}')
            missing_codes.append(code)
            continue
    frames.append(df)

if missing_codes:
    print(f'Skipped {len(missing_codes)} codes: {missing_codes}')

data = pd.concat(frames, ignore_index=True)
data = detect_limit_price(data)
data = detect_suspension(data)
validate_ohlcv(data)

print(f'Data: {len(data)} rows, {data["code"].nunique()} stocks')
print(f'Date range: {data["date"].min().date()} ~ {data["date"].max().date()}')

Loading 100 stocks from 2016-05-24 to 2026-05-24...


Data: 220498 rows, 100 stocks
Date range: 2016-05-24 ~ 2026-05-22


## Step 2: 行业分组（同上一轮）

In [3]:
INDUSTRY_GROUPS = {
    '银行': [
        '601939', '601398', '601288', '601988', '600036',
        '601658', '601328', '601998', '601166', '600000',
        '000001', '601818', '002142', '600919',
    ],
    '非银金融': [
        '601318', '601628', '601601', '601319', '601336',
        '600030', '300059', '601211', '300033',
        '601066', '601688', '601995',
    ],
    '能源资源': [
        '601857', '600938', '600028',
        '601088', '601225', '601898', '600188',
        '603993', '601899', '600111', '601600', '600362',
    ],
    '科技半导体': [
        '688981', '688256', '688041', '002371', '688008',
        '603986', '688012', '002415', '002916', '002463',
        '600183',
    ],
    '通信设备': [
        '600941', '601728',
        '300308', '300502', '300394', '000063',
    ],
    '消费': [
        '600519', '000858', '600809',
        '000651', '000333', '600690',
        '603288', '600887', '002050',
    ],
    '医药': [
        '600276', '603259', '300760',
    ],
    '电力公用': [
        '600900', '600930', '003816', '601985', '600025',
    ],
    '新能源': [
        '300750', '002594', '300274',
        '000792', '002460',
    ],
    '电子制造': [
        '601138', '002475', '002384', '300476', '300433',
        '002938', '300408', '000725', '000338', '600309',
    ],
    '装备制造': [
        '600150', '300124', '600406', '600031', '601100',
        '601766', '302132', '600989', '002714', '601668',
        '601816', '601919', '002352',
    ],
}

all_grouped = set()
for name, codes in INDUSTRY_GROUPS.items():
    all_grouped.update(codes)

all_stocks = set(STOCKS)
missing = all_stocks - all_grouped
extra = all_grouped - all_stocks

print(f'Groups: {len(INDUSTRY_GROUPS)}, Total: {len(all_grouped)}')
if missing:
    print(f'MISSING: {missing}')
if extra:
    print(f'EXTRA (not in data): {extra}')
for name, codes in INDUSTRY_GROUPS.items():
    print(f'  {name}: {len(codes)} stocks')

Groups: 11, Total: 100
  银行: 14 stocks
  非银金融: 12 stocks
  能源资源: 12 stocks
  科技半导体: 11 stocks
  通信设备: 6 stocks
  消费: 9 stocks
  医药: 3 stocks
  电力公用: 5 stocks
  新能源: 5 stocks
  电子制造: 10 stocks
  装备制造: 13 stocks


## Step 3: 运行矩阵回测（55个组合）

In [4]:
STRATEGY_SPECS = [
    {'name': 'reversed_gtja_vwap'},
    {'name': 'gtja_vwap'},
    {'name': 'gtja_momentum'},
    {'name': 'gtja_volume_price'},
    {'name': 'gtja_volatility'},
]

results = run_matrix(
    pool_groups=INDUSTRY_GROUPS,
    strategy_specs=STRATEGY_SPECS,
    data=data,
    capital=1_000_000,
    max_weight=0.3,
)

print(f'Results: {len(results)} rows')
results

Results: 55 rows


,strategy,pool,total_return,annual_return,sharpe_ratio,max_drawdown,win_rate,trade_count
0,reversed_gtja_vwap,银行,0.539338,0.045826,1.724814e-01,0.336218,0.547945,149
1,gtja_vwap,银行,0.504359,0.043332,1.605183e-01,0.328860,0.627273,225
2,gtja_momentum,银行,0.302794,0.027857,9.550607e-02,0.316581,0.476071,799
3,gtja_volume_price,银行,0.691830,0.056138,2.214531e-01,0.298536,0.516224,683
4,gtja_volatility,银行,0.430009,0.037853,1.366032e-01,0.342987,0.452769,619
5,reversed_gtja_vwap,非银金融,0.418433,0.036977,1.418898e-01,0.386128,0.513514,151
6,gtja_vwap,非银金融,0.709687,0.057290,2.323279e-01,0.441910,0.511962,423
7,gtja_momentum,非银金融,0.773757,0.061338,2.433399e-01,0.470588,0.466851,729
8,gtja_volume_price,非银金融,0.407161,0.036118,1.510976e-01,0.446966,0.476584,731
9,gtja_volatility,非银金融,0.564554,0.047592,1.891300e-01,0.408341,0.470990,591


## Step 4: 对比：3年 vs 10年

In [5]:
# 每行业最优策略（按 Sharpe）
best = best_per_pool(results, metric='sharpe_ratio')
print('=== Best Strategy Per Industry (10-year) ===')
print(f'{"Industry":<10} {"Best Strategy":<24} {"Sharpe":>8}')
print('-' * 45)
for _, row in best.iterrows():
    print(f'{row["pool"]:<10} {row["strategy"]:<24} {row["sharpe_ratio"]:>8.2f}')

print()
# 跨期稳定性：比较3年vs10年的最优策略是否一致
print('=== Cross-period Comparison ===')
best_3y = {
    '银行': 'reversed_gtja_vwap',
    '非银金融': 'gtja_vwap',
    '能源资源': 'reversed_gtja_vwap',
    '科技半导体': 'gtja_volatility',
    '通信设备': 'gtja_momentum',
    '消费': 'gtja_momentum',
    '医药': 'gtja_volume_price',
    '电力公用': 'gtja_vwap',
    '新能源': 'gtja_volume_price',
    '电子制造': 'reversed_gtja_vwap',
    '装备制造': 'gtja_vwap',
}

same = 0
total = 0
for _, row in best.iterrows():
    pool = row['pool']
    strat = row['strategy']
    old = best_3y.get(pool, '?')
    match = '✓' if strat == old else f'✗ (was {old})'
    total += 1
    if strat == old:
        same += 1
    print(f'  {pool:<10} 3y={old:<22} 10y={strat:<22} {match}')

print(f'\nStable: {same}/{total} industries')

=== Best Strategy Per Industry (10-year) ===
Industry   Best Strategy              Sharpe
---------------------------------------------
医药         gtja_volume_price            0.08
新能源        gtja_vwap                    0.42
消费         gtja_vwap                    0.49
电力公用       gtja_vwap                    0.20
电子制造       gtja_volatility              0.50
科技半导体      gtja_momentum                0.61
能源资源       reversed_gtja_vwap           0.47
装备制造       gtja_momentum                0.40
通信设备       gtja_volatility              0.46
银行         gtja_volume_price            0.22
非银金融       gtja_momentum                0.24

=== Cross-period Comparison ===
  医药         3y=gtja_volume_price      10y=gtja_volume_price      ✓
  新能源        3y=gtja_volume_price      10y=gtja_vwap              ✗ (was gtja_volume_price)
  消费         3y=gtja_momentum          10y=gtja_vwap              ✗ (was gtja_momentum)
  电力公用       3y=gtja_vwap              10y=gtja_vwap              ✓
  电子制造       3y=reve

## Step 5: Sharpe 矩阵（10年）

In [6]:
sharpe_pivot = pivot_matrix(results, metric='sharpe_ratio')

fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(sharpe_pivot.values, aspect='auto', cmap='RdYlGn')

ax.set_xticks(range(len(sharpe_pivot.columns)))
ax.set_xticklabels(sharpe_pivot.columns, rotation=45, ha='right')
ax.set_yticks(range(len(sharpe_pivot.index)))
ax.set_yticklabels(sharpe_pivot.index)

for i in range(len(sharpe_pivot.index)):
    for j in range(len(sharpe_pivot.columns)):
        val = sharpe_pivot.iloc[i, j]
        text = f'{val:.2f}' if not pd.isna(val) and abs(val) < 100 else '—'
        bg = im.norm(val) if not pd.isna(val) and abs(val) < 100 else 0.5
        color = 'white' if bg < 0.4 or bg > 0.6 else 'black'
        ax.text(j, i, text, ha='center', va='center', fontsize=9, color=color)

ax.set_title('Strategy × Industry — Sharpe Ratio (10Y: 2016-2026)', fontsize=14, fontweight='bold')
fig.colorbar(im, ax=ax, shrink=0.8, label='Sharpe')
fig.tight_layout()

output_dir = Path('output')
output_dir.mkdir(exist_ok=True)
fig.savefig(output_dir / 'industry_matrix_sharpe_10y.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: industry_matrix_sharpe_10y.png')
sharpe_pivot

Saved: industry_matrix_sharpe_10y.png


/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24798/415835409.py:21: UserWarning: Glyph 21307 (\N{CJK UNIFIED IDEOGRAPH-533B}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24798/415835409.py:21: UserWarning: Glyph 33647 (\N{CJK UNIFIED IDEOGRAPH-836F}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24798/415835409.py:21: UserWarning: Glyph 26032 (\N{CJK UNIFIED IDEOGRAPH-65B0}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24798/415835409.py:21: UserWarning: Glyph 33021 (\N{CJK UNIFIED IDEOGRAPH-80FD}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1rj72ddzftb81k440000gn/T/ipykernel_24798/415835409.py:21: UserWarning: Glyph 28304 (\N{CJK UNIFIED IDEOGRAPH-6E90}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/var/folders/_7/p1ny6h2j1

strategy,gtja_momentum,gtja_volatility,gtja_volume_price,gtja_vwap,reversed_gtja_vwap
pool,,,,,
医药,0.054492,0.054492,0.077646,0.054492,-4.648143e+16
新能源,0.419349,0.419349,0.389697,0.419349,-4.648143e+16
消费,0.392250,0.392167,0.309918,0.491796,3.782273e-01
电力公用,0.196943,0.196943,0.153458,0.196943,-4.648143e+16
电子制造,0.455563,0.504101,0.494629,0.474464,4.133792e-01
科技半导体,0.609604,0.516527,0.547037,0.548758,5.158318e-01
能源资源,0.427040,0.452139,0.431085,0.389450,4.742589e-01
装备制造,0.395384,0.341447,0.286620,0.255043,2.175223e-01
通信设备,0.454583,0.461327,0.452628,0.449009,1.093838e-01


## Step 6: 策略排名（10年）

In [7]:
results['sharpe_rank'] = results.groupby('pool')['sharpe_ratio'].rank(ascending=False)

ranking = results.groupby('strategy').agg(
    avg_rank=('sharpe_rank', 'mean'),
    win_count=('sharpe_rank', lambda x: (x == 1).sum()),
    avg_sharpe=('sharpe_ratio', lambda x: x[x.abs() < 100].mean()),
    avg_annual_return=('annual_return', 'mean'),
    avg_max_dd=('max_drawdown', 'mean'),
).sort_values('avg_rank')

ranking

,avg_rank,win_count,avg_sharpe,avg_annual_return,avg_max_dd
strategy,,,,,
gtja_momentum,2.454545,3,0.340369,0.094090,0.450002
gtja_volatility,2.454545,2,0.333111,0.093603,0.453130
gtja_vwap,2.818182,1,0.333832,0.090188,0.441733
gtja_volume_price,3.000000,2,0.319570,0.087838,0.456103
reversed_gtja_vwap,4.272727,1,0.302872,0.061518,0.289747


## Step 7: 三年 vs 十年对比总结

（运行后填写）